# 🔔 Anomaly Detection — Manufacturing Quality Control

Uses **SynapseML** anomaly detection on sensor telemetry and production batch data.

**Detects:**
- Sensor anomalies: temperature, pressure, vibration out of expected range
- Defect rate spikes: unusual defect counts per production batch

**Reads from:** `silver_sensor_readings`, `silver_production_batches`

**Writes to:** `gold_sensor_anomalies`, `gold_defect_anomalies`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, stddev,
    window, count, sum as spark_sum, expr, to_date, hour,
    lag, abs as spark_abs, percentile_approx
)
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

## 1. Load Silver data

In [ ]:
sensor_df = spark.read.table('silver_sensor_readings')
batch_df = spark.read.table('silver_production_batches')

print(f'Silver sensor readings: {sensor_df.count()} rows')
print(f'Silver production batches: {batch_df.count()} rows')

## 2. Sensor Anomaly Detection with SynapseML

Use SynapseML's `IsolationForest` for multivariate anomaly detection on sensor features (temperature, pressure, vibration, humidity).

In [ ]:
# Prepare feature vector for anomaly detection
sensor_features = ['temperature', 'pressure', 'vibration', 'humidity']

# Drop nulls in feature columns
sensor_clean = sensor_df.dropna(subset=sensor_features)

assembler = VectorAssembler(
    inputCols=sensor_features,
    outputCol='features',
    handleInvalid='skip'
)
sensor_vec = assembler.transform(sensor_clean)
print(f'Vectorized sensor readings: {sensor_vec.count()} rows')

In [ ]:
from synapse.ml.isolationforest import IsolationForest

# Train Isolation Forest model
iso_forest = IsolationForest(
    featuresCol='features',
    predictionCol='is_anomaly',
    scoreCol='anomaly_score',
    contamination=0.05,  # expect ~5% anomalies
    numEstimators=100,
    maxSamples=256,
    maxFeatures=1.0,
    bootstrap=False
)

model = iso_forest.fit(sensor_vec)
sensor_scored = model.transform(sensor_vec)

anomaly_count = sensor_scored.filter(col('is_anomaly') == 1).count()
total = sensor_scored.count()
print(f'Sensor anomalies detected: {anomaly_count} / {total} ({anomaly_count/total*100:.1f}%)')

In [ ]:
# Determine which sensor channels are anomalous using per-machine statistics
w = Window.partitionBy('machine_id')

sensor_anomalies = (
    sensor_scored
    .withColumn('temp_mean', avg('temperature').over(w))
    .withColumn('temp_std', stddev('temperature').over(w))
    .withColumn('press_mean', avg('pressure').over(w))
    .withColumn('press_std', stddev('pressure').over(w))
    .withColumn('vib_mean', avg('vibration').over(w))
    .withColumn('vib_std', stddev('vibration').over(w))
    .withColumn('temp_zscore', spark_abs((col('temperature') - col('temp_mean')) / col('temp_std')))
    .withColumn('press_zscore', spark_abs((col('pressure') - col('press_mean')) / col('press_std')))
    .withColumn('vib_zscore', spark_abs((col('vibration') - col('vib_mean')) / col('vib_std')))
    .withColumn('anomaly_channel',
        when(col('temp_zscore') > 2.5, 'temperature')
        .when(col('press_zscore') > 2.5, 'pressure')
        .when(col('vib_zscore') > 2.5, 'vibration')
        .otherwise('multivariate')
    )
    .withColumn('severity',
        when(col('anomaly_score') > 0.7, 'critical')
        .when(col('anomaly_score') > 0.5, 'warning')
        .otherwise('info')
    )
    .withColumn('detection_timestamp', current_timestamp())
)

In [ ]:
# Write Gold sensor anomaly table
gold_sensor_anomalies = (
    sensor_anomalies
    .select(
        'reading_id', 'machine_id', 'reading_timestamp',
        'temperature', 'pressure', 'vibration', 'humidity',
        'is_anomaly', 'anomaly_score', 'anomaly_channel', 'severity',
        'shift', 'reading_date', 'reading_hour',
        'detection_timestamp'
    )
)

gold_sensor_anomalies.write.mode('overwrite').format('delta').saveAsTable('gold_sensor_anomalies')
print(f'Gold sensor anomalies: {gold_sensor_anomalies.count()} rows')
print(f'  - Anomalies: {gold_sensor_anomalies.filter(col("is_anomaly") == 1).count()}')
print(f'  - Normal: {gold_sensor_anomalies.filter(col("is_anomaly") == 0).count()}')

## 3. Defect Rate Anomaly Detection

Detect unusual defect rate spikes per production line using IQR-based detection + SynapseML.

In [ ]:
from pyspark.sql.functions import datediff

# Calculate per-batch metrics
batch_metrics = (
    batch_df
    .withColumn('defect_rate', col('defect_count') / col('units_produced') * 100)
    .withColumn('yield_pct', (col('units_produced') - col('defect_count')) / col('units_produced') * 100)
    .withColumn('downtime_pct', col('downtime_minutes') / 480 * 100)  # 8-hour shift
)

# Compute IQR per production line for defect rate
line_stats = (
    batch_metrics
    .groupBy('production_line')
    .agg(
        avg('defect_rate').alias('mean_defect_rate'),
        stddev('defect_rate').alias('std_defect_rate'),
        percentile_approx('defect_rate', 0.25).alias('q1_defect_rate'),
        percentile_approx('defect_rate', 0.75).alias('q3_defect_rate'),
        avg('downtime_pct').alias('mean_downtime_pct'),
        stddev('downtime_pct').alias('std_downtime_pct')
    )
    .withColumn('iqr', col('q3_defect_rate') - col('q1_defect_rate'))
    .withColumn('upper_fence', col('q3_defect_rate') + 1.5 * col('iqr'))
)

# Join and flag anomalies
defect_anomalies = (
    batch_metrics.join(line_stats, 'production_line')
    .withColumn('defect_zscore',
        spark_abs((col('defect_rate') - col('mean_defect_rate')) / col('std_defect_rate'))
    )
    .withColumn('downtime_zscore',
        spark_abs((col('downtime_pct') - col('mean_downtime_pct')) / col('std_downtime_pct'))
    )
    .withColumn('is_defect_anomaly',
        when(
            (col('defect_rate') > col('upper_fence')) | (col('defect_zscore') > 2.5),
            lit(True)
        ).otherwise(lit(False))
    )
    .withColumn('is_downtime_anomaly',
        when(col('downtime_zscore') > 2.5, lit(True)).otherwise(lit(False))
    )
    .withColumn('anomaly_type',
        when(col('is_defect_anomaly') & col('is_downtime_anomaly'), 'defect+downtime')
        .when(col('is_defect_anomaly'), 'defect_spike')
        .when(col('is_downtime_anomaly'), 'downtime_spike')
        .otherwise('normal')
    )
    .withColumn('severity',
        when(col('defect_zscore') > 3.0, 'critical')
        .when(col('defect_zscore') > 2.0, 'warning')
        .otherwise('info')
    )
    .withColumn('detection_timestamp', current_timestamp())
)

print(f'Defect anomalies: {defect_anomalies.filter(col("is_defect_anomaly")).count()} batches')
print(f'Downtime anomalies: {defect_anomalies.filter(col("is_downtime_anomaly")).count()} batches')

In [ ]:
# Write Gold defect anomaly table
gold_defect_anomalies = (
    defect_anomalies
    .select(
        'batch_id', 'production_line', 'production_date', 'shift',
        'units_produced', 'defect_count', 'defect_rate', 'yield_pct',
        'downtime_minutes', 'downtime_pct',
        'is_defect_anomaly', 'is_downtime_anomaly', 'anomaly_type', 'severity',
        'defect_zscore', 'downtime_zscore',
        'mean_defect_rate', 'upper_fence',
        'detection_timestamp'
    )
)

gold_defect_anomalies.write.mode('overwrite').format('delta').saveAsTable('gold_defect_anomalies')
print(f'Gold defect anomalies written: {gold_defect_anomalies.count()} rows')

In [ ]:
# Optimize Gold tables
spark.sql('OPTIMIZE gold_sensor_anomalies')
spark.sql('OPTIMIZE gold_defect_anomalies')
print('Gold anomaly tables optimized')

In [ ]:
# Summary
print('\n=== Anomaly Detection Summary ===')
print(f'Sensor anomalies: {sensor_scored.filter(col("is_anomaly") == 1).count()} readings flagged')
print(f'Defect anomalies: {defect_anomalies.filter(col("is_defect_anomaly")).count()} batches flagged')
print(f'Downtime anomalies: {defect_anomalies.filter(col("is_downtime_anomaly")).count()} batches flagged')
print('\nTables created:')
print('  - gold_sensor_anomalies')
print('  - gold_defect_anomalies')